In [139]:
import pandas as pd
import numpy as np

In [140]:
df = pd.read_csv('data/02_telco_cleaned.csv')

# **Data Transformation** 

## 1. **Feature Selection**

<small>The objective here is to take our cleaned columns and add, drop or construct new features that make it easier for a machine learning algorithm to spot patterns (like high churn risk).

##### **1.a) Dropping the Gender Column**

In [141]:
#Confirming the number of columns and rows in the dataframe
#The number of rows was 7,032 and columns was 20 after the data cleaning process.
print(df.shape)
print("The number of rows is: ", df.shape[0])
print("The number of columns is: ", df.shape[1])

(7032, 20)
The number of rows is:  7032
The number of columns is:  20


In [142]:
#Here we can check the columns of the dataframe to see what we have in it.
print(df.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')


<small> 

In the **Exploratory Data Analysis** stage, we discovered that the churn behaviour is nearly identical regardless of **'gender'**. The churn rates were as follows:

**Female customers:** ~26.9% churn rate

**Male customers:** ~26.2% churn rate

(Please refer to the chart 05_categorical_feature_analysis.png for visual representation.)

Therefore, this feature does not hold any predictive power on a machine learning model. I am going to drop it as this will simplify my data and reduce noise ensuring that the model is not picking up on random fluctuations.

In [143]:
# Dropping the 'gender' column as it is not needed for the analysis
df.drop('gender', axis=1, inplace=True)

#Verifying if the column has been dropped
print("New number of columns:", len(df.columns)) 

New number of columns: 19


##### **1.b) Dropping the TotalCharges Column**

<small> At the EDA stage, under the section 2.b) Numerical Analysis, we discovered a multicollinearity relationship between tenure and TotalCharges. 

**Correlation with tenure:** 0.83 (Extremely high)

**Correlation with MonthlyCharges:** 0.65 (High)

Linear models (like Logistic Regression) struggle with multicollineatry relationships because it makes it difficult for the model to estimate the independent effect of each variable.

**TotalCharges is essentially the mathematical product of tenure (months customer has been with the compnay) multiplied by MonthlyCharges.**


So the column TotalCharges will be dropped also to reduce noise in the model. However, tenure and MonthtlyCharges will be kept.



In [144]:
df.drop('TotalCharges', axis=1, inplace=True)

print("New number of columns is now:", len(df.columns)) 

New number of columns is now: 18


## 2. **Categorical Encoding**

<small>Because machine learning frameworks operate purely on numerical data, an encoding pipeline was implemented to translate qualitative text categories into quantitative values.

In [145]:
#Here is a preview of the data types of each column in the dataframe. 
# This will help us understand what kind of data we are working with.
# We then can transform the data for modeling purposes.
df.dtypes

SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
Churn                   str
dtype: object

<small>The only numerical columns are SeniorCitizen, tenure and MonthlyCharges. The rest are text values which will be transfomed into numbers in following steps. 

### **2.a) Target Variable Encoding**

<small>The target variable is **Churn** and is represented by the string values 'Yes' or 'No'. These must be converted to '1' or '0's inorder to comply with the machine learning frameworks.

In [146]:
#Encoding the target variable 'Churn' using a dictionary map
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Verifying the transformation
print("Value counts after encoding:")
print(df['Churn'].value_counts())

Value counts after encoding:
Churn
0    5163
1    1869
Name: count, dtype: int64


<small> Successfully converted Churn string values (Yes and No) to a binary numeric format (1 and 0)

### **2.b) Binary Encoding**

<small>We have columns with exactly two unique string values and these can be converted into a binary numeric format as shown above for the Target Variable Encoding.

In [147]:
# Viewing all the columns in the dataframe to see the string values
# Picking the binary columns to map them to numerical values for modeling purposes
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,0
1,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,0
2,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,1
3,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,0
4,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,1


In [ ]:
# Here we can check the unique values in each of the object type columns
# We can then see what kind of data we have in them.
#We can then decide how to encode them for modeling purposes.
for col in df.select_dtypes(include=['object']).columns:
    print(f"--- Unique values in '{col}' ---")
    print(df[col].unique())
    print()  

--- Unique values in 'Partner' ---
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

--- Unique values in 'Dependents' ---
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

--- Unique values in 'PhoneService' ---
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

--- Unique values in 'MultipleLines' ---
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

--- Unique values in 'InternetService' ---
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

--- Unique values in 'OnlineSecurity' ---
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

--- Unique values in 'OnlineBackup' ---
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

--- Unique values in 'DeviceProtection' ---
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

--- Unique values in 'TechSupport' ---
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

--- Unique values in 'StreamingTV' ---
<StringArray>

C:\Users\User\AppData\Local\Temp\ipykernel_19484\3881225077.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [ ]:
# Manual clean mapping for binary columns
numerical_mappings = {
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1},
}

for col, mapping in numerical_mappings.items():
    df[col] = df[col].map(mapping)

In [150]:
#One hot encoding for categorical columns
one_hot_columns = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'PaymentMethod', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV','StreamingMovies']

df = pd.get_dummies(df, columns=one_hot_columns, drop_first=True)

In [151]:
#Convert all True/False columns to 1/0 automatically
df = df.astype({col: int for col in df.select_dtypes(include='bool').columns})

In [152]:
df.dtypes

SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
Contract                                   int64
PaperlessBilling                           int64
MonthlyCharges                           float64
Churn                                      int64
MultipleLines_No phone service             int64
MultipleLines_Yes                          int64
InternetService_Fiber optic                int64
InternetService_No                         int64
OnlineSecurity_No internet service         int64
OnlineSecurity_Yes                         int64
PaymentMethod_Credit card (automatic)      int64
PaymentMethod_Electronic check             int64
PaymentMethod_Mailed check                 int64
OnlineBackup_No internet service           int64
OnlineBackup_Yes                           int64
DeviceProtection_No 

In [153]:
df.to_csv("data/03_telco_transformed.csv", index=False)